This is the file where the structure of the HRM should be defined. This includes the training loop and the __init__ and forward of the network. It should not include the actual High and Low Level architectures, which are encoder-only transformers.

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import RandAugment
import transformers
from transformers import BertConfig, BertModel, BertConfig
from transformers.models.bert.modeling_bert import BertEncoder

import random

from tqdm import tqdm
import sys


from google.colab import drive

torch.manual_seed(0)

drive.mount('/content/drive', force_remount=True)


base_dir = "/content/drive/MyDrive/Junior/Second Semester/CS 4782 - DL/Final Project - HRM"
sys.path.append(base_dir)
from Model.HRM_Model import HRM
from Model.HRM_Components import Encoder, HighLevel, LowLevel, Head

sys.path.append(base_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("meta")
print(F"Device set to {device}")

Mounted at /content/drive
Device set to cuda


In [6]:
from datasets import load_dataset
import torch
from torch.utils.data import Dataset, DataLoader
train_size = 1000
test_size = 200


ds = load_dataset("sapientinc/sudoku-extreme", split="train[:" + str(train_size) + "]")
ds_test = load_dataset("sapientinc/sudoku-extreme", split="test[:" + str(test_size) + "]")
print(ds.column_names)


def encode(example):
    puzzle = [int(c) if c != '.' else 0 for c in example['question']]
    solution = [int(c) for c in example['answer']]
    return {"x": puzzle, "y": solution}

ds = ds.map(encode)
ds_test = ds_test.map(encode)


class SudokuDataset(Dataset):
    def __init__(self, ds):
        self.ds = ds
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        item = self.ds[idx]
        return torch.tensor(item["x"], dtype=torch.long), torch.tensor(item["y"], dtype=torch.long)

train_dataset = SudokuDataset(ds)
test_dataset = SudokuDataset(ds)

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True)





/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/719M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/79.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3831994 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/422786 [00:00<?, ? examples/s]

['source', 'question', 'answer', 'rating']


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [7]:
# #Create Temporary trivial modules to simulate the high and low level structure
# class TempHighLevel(nn.Module):
#   def __init__(self, d_model=32):
#     super().__init__()
#     self.fc1 = nn.Linear(d_model, d_model)
#     self.norm1 = nn.LayerNorm(d_model)
#     self.fc2 = nn.Linear(d_model, d_model)
#     self.norm2 = nn.LayerNorm(d_model)


#   def forward(self, x, z_H, z_L):
#     x = self.fc1(x)
#     x = self.norm1(x)
#     x = F.relu(x)
#     x = self.fc2(x)
#     x = self.norm2(x)
#     x = F.relu(x)
#     return x


# class TempLowLevel(nn.Module):
#   def __init__(self, d_model=32):
#     super().__init__()
#     self.fc1 = nn.Linear(d_model, d_model)
#     self.norm1 = nn.LayerNorm(d_model)
#     self.fc2 = nn.Linear(d_model, d_model)
#     self.norm2 = nn.LayerNorm(d_model)


#   def forward(self, x, z_H, z_L):
#     x = self.fc1(x)
#     x = self.norm1(x)
#     x = F.relu(x)
#     x = self.fc2(x)
#     x = self.norm2(x)
#     x = F.relu(x)
#     return x



# class TempEncoder(nn.Module):
#   def __init__(self, vocab_size=10, d_model=32):
#     super().__init__()
#     self.embedding = nn.Embedding(vocab_size, d_model)  # convert token -> vector

#   def forward(self, x):
#     return self.embedding(x)  # (B, L) -> (B, L, d_model)

# class TempHead(nn.Module):
#   def __init__(self, d_model=32, vocab_size=10):
#     super().__init__()
#     self.linear = nn.Linear(d_model, vocab_size)

#   def forward(self, x):
#     return self.linear(x)  # (B, L, vocab_size)


# ---------- Embedder (token -> latent) ----------
class TempEncoder(nn.Module):
    def __init__(self, vocab_size=10, d_model=32, max_len=81):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        # optional learned positional encoding
        self.pos_embedding = nn.Parameter(torch.zeros(1, max_len, d_model))

    def forward(self, x):
        # x: (B, L)
        x = self.embedding(x) + self.pos_embedding[:, :x.size(1), :]
        return x  # (B, L, d_model)



# ---------- Latent-only High-Level module ----------
class TempHighLevel(nn.Module):
    def __init__(self, d_model=32, n_layers=2, n_heads=4, intermediate_size=64):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=intermediate_size,
            batch_first=True,  # keep (B, L, d_model)
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, x, z_H,z_L):
        # x: (B, L, d_model), ignores extra arguments
        return self.encoder(x+z_H + z_L)  # (B, L, d_model) -> latent -> latent

# ---------- Latent-only Low-Level module ----------
class TempLowLevel(nn.Module):
    def __init__(self, d_model=32, n_layers=2, n_heads=4, intermediate_size=64):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=intermediate_size,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, x, z_H, z_L):
        return self.encoder(x+z_H+z_L)  # (B, L, d_model) -> latent -> latent
# ---------- Unembedder (latent -> token) ----------
class TempHead(nn.Module):
    def __init__(self, d_model=32, vocab_size=10):
        super().__init__()
        self.linear = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        return self.linear(x)  # (B, L, vocab_size)

In [8]:
#The module for HRM. Assumes that the high level and low level architectures are implemented
class HRM(nn.Module):
  def __init__(self, L_module, H_module, encoder, head, M, N, T):
    super().__init__()
    assert M > 0
    assert T > 0
    self.low_level = L_module
    self.high_level = H_module
    self.M = M
    self.N = N
    self.T = T
    self.head = head
    self.encoder = encoder

  # def forward(self, x_embed, z_H, z_L):
  #   # All segments except last — no grad
  #   x_embed = self.encoder(x_embed)
  #   with torch.no_grad():
  #     for m in range(self.M - 1):
  #       for t in range(self.T):
  #         z_L = self.low_level(x_embed, z_H, z_L)
  #       z_H = self.high_level(x_embed, z_H, z_L)

  #   # Last segment — grad flows here only
  #   # no detach needed — z_H and z_L already have no graph
  #   for t in range(self.T):
  #     z_L = self.low_level(x_embed, z_H, z_L)
  #   z_H = self.high_level(x_embed, z_H, z_L)

  # return self.head(z_H)

  def encode(self, x):
      return self.encoder(x)

  def step(self, z_H, z_L, x_embed):
    """One full segment — T low passes then one high pass.
    Input states should already be detached by the caller."""
    with torch.no_grad():
      #Do N-1 High Passes
      for n in range(self.N - 1):
        for t in range(self.T):
          z_L = self.low_level(x_embed, z_H, z_L)
        z_H = self.high_level(x_embed, z_H, z_L)  # grad flows only here

      #Do the final High pass. Only take gradient at the end
      for t in range(self.T - 1):
        z_L = self.low_level(x_embed, z_H, z_L)
    z_L = self.low_level(x_embed, z_H, z_L)     # last L — grad flows
    z_H = self.high_level(x_embed, z_H, z_L)  # grad flows only here
    return z_H, z_L

  def predict(self, x):
    if len(x.shape) == 1:
        x = x.unsqueeze(0)
    x = x.long()  # ensure indices are integer
    x = x.to(next(self.parameters()).device)  # move to same device as model
    out = self.forward(x)
    labels = out.argmax(dim=-1)
    return labels


  def forward(self, x):
    x_embed = self.encode(x)
    B, L, d = x_embed.shape
    z_H = torch.zeros(B, L, d, device=x.device)
    z_L = torch.zeros(B, L, d, device=x.device)

    for m in range(self.M):
      z_H, z_L = self.step(z_H, z_L.detach(), x_embed)


    return self.head(z_H)

In [9]:
#Define the optimizer and scheduler used
# Define the triplet model
d_model = 512
M = 1
N = 2
T = 2
n_layers = 4
n_heads = 4
vocab_size = 10
lr = 1e-4
num_epochs = 5

high_level = HighLevel(d_model=d_model, n_layers=n_layers, n_heads=n_heads, intermediate_size=4*d_model)
low_level = HighLevel(d_model=d_model, n_layers=n_layers, n_heads=n_heads, intermediate_size=4*d_model)
head = Head(vocab_size=vocab_size, d_model=d_model)
encoder = Encoder(vocab_size=vocab_size, d_model=d_model, max_len=81)


HRM_model = HRM(low_level, high_level, encoder, head, M, N, T).to(device)

# Define the optimizer
optimizer = optim.AdamW(HRM_model.parameters(), lr=lr)

# Define the learning rate scheduler
from transformers import get_cosine_schedule_with_warmup

num_training_steps = len(train_dataloader) * num_epochs
num_warmup_steps = int(0.05 * num_training_steps)  # 5% warmup
"""
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)
scheduler = transformers.get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=1 * len(train_dataloader))
"""
scheduler = None

In [10]:
# def train_epoch(model, train_loader, optimizer, loss_fn, NUM_EPOCHS):
#     for epoch in range(NUM_EPOCHS):
#       model.train()
#       total_loss = 0

#       for x, y in tqdm(train_loader):
#           x_embed = model.encode(x)
#           B, L, d = x_embed.shape
#           z_H = torch.zeros(B, L, d, device=x.device)
#           z_L = torch.zeros(B, L, d, device=x.device)

#           for m in range(model.M):
#               z_H, z_L = model.step(z_H, z_L, x_embed)

#               loss = loss_fn(model.head(z_H), y)
#               optimizer.zero_grad()
#               loss.backward()
#               optimizer.step()

#               z_H = z_H.detach()
#               z_L = z_L.detach()

#               total_loss += loss.item()

#       avg_loss = total_loss / (len(train_loader) * model.M)
#       print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}")

#     return total_loss / (len(loader) * model.M)
# train_epoch(HRM_model,train_dataloader, optimizer,nn.CrossEntropyLoss(), 1)


In [ ]:
def train_hrm_deepsup(model, loader, optimizer, loss_fn, scheduler=None, num_epochs=10, vocab_size=10):
    """
    Train HRM with deep supervision for multiple epochs and report stats.

    model      : HRM model
    loader     : DataLoader with (input_seq, target_seq)
    optimizer  : optimizer
    loss_fn    : loss function (nn.CrossEntropyLoss)
    scheduler  : optional learning rate scheduler
    num_epochs : number of epochs to train
    vocab_size : size of discrete token vocabulary
    """
    print(f"Number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        correct_tokens = 0
        total_tokens = 0

        for x, y in tqdm(loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            x = x.to(device)
            y = y.to(device)
            x_embed = model.encode(x)  # trainable encoder
            B, L, d = x_embed.shape
            z_H = torch.zeros(B, L, d, device=x.device)
            z_L = torch.zeros(B, L, d, device=x.device)

            for m in range(model.M):
                # Detach x_embed after first segment so encoder only trained once
                x_in = x_embed if m == 0 else x_embed.detach()
                z_H, z_L = model.step(z_H, z_L, x_in)

                # Segment-wise forward + loss
                y_pred = model.head(z_H)               # (B, L, vocab_size)
                y_pred_flat = y_pred.view(B * L, vocab_size)
                y_flat = y.view(-1)                     # (B*L,)

                y_train = y.clone()
                y_train[x != 0] = -100                 # ignore givens, train only on blanks
                y_train_flat = y_train.view(-1)

                loss = loss_fn(y_pred_flat, y_train_flat)

                #loss = loss_fn(y_pred_flat, y_flat)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()

                # Detach hidden states for next segment
                z_H = z_H.detach()
                z_L = z_L.detach()

                total_loss += loss.item()

                # Compute token-level accuracy
                y_pred_labels = y_pred_flat.argmax(dim=1)
                correct_tokens += (y_pred_labels == y_flat).sum().item()
                total_tokens += y_flat.numel()

        avg_loss = total_loss / (len(loader) * model.M)
        acc = correct_tokens / total_tokens

        print(f"Epoch {epoch+1}: Avg Loss = {avg_loss:.4f}, Token Accuracy = {acc*100:.2f}%")

    return model
trained_model = train_hrm_deepsup(HRM_model,train_dataloader, optimizer,nn.CrossEntropyLoss(ignore_index=-100), scheduler, num_epochs=num_epochs, vocab_size=vocab_size).eval()
print(trained_model)

Number of trainable parameters: 25,270,794


Epoch 1/5: 100%|██████████| 16/16 [00:07<00:00,  2.06it/s]


Epoch 1: Avg Loss = 2.2759, Token Accuracy = 11.31%


Epoch 2/5: 100%|██████████| 16/16 [00:07<00:00,  2.02it/s]


Epoch 2: Avg Loss = 2.2070, Token Accuracy = 11.34%


Epoch 3/5: 100%|██████████| 16/16 [00:08<00:00,  1.97it/s]


Epoch 3: Avg Loss = 2.2041, Token Accuracy = 11.41%


Epoch 4/5:  19%|█▉        | 3/16 [00:01<00:06,  1.88it/s]

In [ ]:
#Visualize a sudoku
x,y = train_dataset[random.randint(0,train_size-1)]
x = torch.tensor(x, dtype=torch.long)
print("X: " + str(x))
print("Prediction: " + str(trained_model.predict(x)))
print("Y: " + str(y))
